# 05. Simulación A/B: Estrategia inteligente vs. línea base

## Objetivo
Cuantificar la ganancia de eficiencia del algoritmo **inteligente** (demanda no cubierta) frente a una **línea base ponderada por población**.

## Escenarios
- **Estrategia A (Línea base):** distribuye $N$ cargadores proporcional a la **densidad de vehículos**. Simula una política estándar de inversión pública.
- **Estrategia B (Inteligente):** K-Means ponderado por **demanda no cubierta**. Prioriza zonas con alta demanda y baja oferta actual.

## Métricas
- **VE servidos (ponderado):** para cada cargador se crea un buffer de 500 m; se calcula el % del área de cada barrio cubierta por algún buffer y se pondera el número de VE por ese porcentaje.
- **Demanda no cubierta capturada (ponderada):** misma lógica de cobertura aplicada a `Unmet_Demand`.
- **Eficiencia de cobertura:** `ve_servidos / total_ve_ciudad`.
- **Ganancia de eficiencia:** mejora porcentual de la estrategia inteligente sobre la línea base.

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from shapely.geometry import Point
import random
import matplotlib.pyplot as plt
import os

DATA_PATH = '../data/processed/barrios_with_demand.geojson'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/processed/barrios_with_demand.geojson'

gdf = gpd.read_file(DATA_PATH)

# Centroides en CRS proyectado (EPSG:25831, UTM 31N) → EPSG:4326
gdf_proj = gdf.to_crs('EPSG:25831')
centroids_4326 = gpd.GeoDataFrame(
    geometry=gdf_proj.geometry.centroid, crs='EPSG:25831'
).to_crs('EPSG:4326')
gdf['lat'] = centroids_4326.geometry.y
gdf['lng'] = centroids_4326.geometry.x

SUPPLY_IMPACT = 80
gdf['Norm_Supply'] = MinMaxScaler().fit_transform(gdf[['Charger_Count']].fillna(0))
gdf['Unmet_Demand'] = (gdf['Demand_Score'] - gdf['Norm_Supply'] * SUPPLY_IMPACT).clip(lower=0)

print(f"Cargados {len(gdf)} barrios.")

Cargados 73 barrios.


## Funciones de simulación

In [2]:
def generate_population_weighted_locations(n, gdf, rng=None, py_rng=None):
    """Genera N ubicaciones proporcionales al padrón de vehículos por barrio."""
    if rng is None:
        rng = np.random.default_rng()
    if py_rng is None:
        py_rng = random.Random()

    weights = gdf['Total_Vehicles'].fillna(0)
    weights = weights / weights.sum()
    sampled_indices = rng.choice(gdf.index.to_numpy(), size=n, replace=True, p=weights.to_numpy())

    points = []
    for idx in sampled_indices:
        poly = gdf.loc[idx, 'geometry']
        min_x, min_y, max_x, max_y = poly.bounds
        while True:
            p = Point(py_rng.uniform(min_x, max_x), py_rng.uniform(min_y, max_y))
            if poly.contains(p):
                points.append([p.y, p.x])
                break

    return np.array(points)


def generate_smart_locations(n, df_features, weights):
    """Ejecuta K-Means ponderado para encontrar las ubicaciones óptimas."""
    n_clusters = min(n, len(df_features))
    if n_clusters < n:
        print(f"Aviso: se solicitaron {n} hubs pero solo hay {len(df_features)} barrios. Usando {n_clusters}.")
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    kmeans.fit(df_features, sample_weight=weights)
    return kmeans.cluster_centers_


def evaluate_coverage(locations, gdf_target, radius_m=500):
    """
    Evalúa la cobertura de un conjunto de cargadores sobre la población de VE.

    Para cada cargador crea un buffer de `radius_m` metros (EPSG:3857), calcula
    el % del área de cada barrio cubierta y pondera el número de VE por ese
    porcentaje. Cada barrio se contabiliza una sola vez (máximo coverage).
    """
    total_evs_city = float(gdf_target['EV_Count'].fillna(0).sum())
    if len(locations) == 0:
        return {
            'served_barrios_count': 0,
            'ev_population_served': 0.0,
            'unmet_demand_captured': 0.0,
            'coverage_efficiency': 0.0,
        }

    barrios = gdf_target[['Barri_ID', 'EV_Count', 'Unmet_Demand', 'geometry']].copy()
    barrios_m = barrios.to_crs(epsg=3857)
    barrio_area = barrios_m.geometry.area.replace(0, np.nan)

    locs_df = pd.DataFrame(locations, columns=['lat', 'lng'])
    loc_gdf = gpd.GeoDataFrame(
        locs_df,
        geometry=gpd.points_from_xy(locs_df['lng'], locs_df['lat']),
        crs=gdf_target.crs
    ).to_crs(epsg=3857)
    buffers = loc_gdf.geometry.buffer(radius_m)

    max_coverage = np.zeros(len(barrios_m), dtype=float)
    for buf in buffers:
        coverage_pct = (
            (barrios_m.geometry.intersection(buf).area / barrio_area)
            .fillna(0).clip(lower=0, upper=1).to_numpy()
        )
        max_coverage = np.maximum(max_coverage, coverage_pct)

    ev_counts = barrios['EV_Count'].fillna(0).to_numpy(dtype=float)
    unmet_vals = barrios['Unmet_Demand'].fillna(0).to_numpy(dtype=float)
    ev_served = float((ev_counts * max_coverage).sum())

    return {
        'served_barrios_count': int((max_coverage > 0).sum()),
        'ev_population_served': ev_served,
        'unmet_demand_captured': float((unmet_vals * max_coverage).sum()),
        'coverage_efficiency': ev_served / total_evs_city if total_evs_city else 0.0,
    }

## Bucle de simulación

In [3]:
SCENARIOS = [10, 25, 50]
BASELINE_ITERATIONS = 300
BASELINE_SEED = 42
RADIUS_M = 500

results = []
locations_export = []

X_coords = gdf[['lat', 'lng']].values
W_weights = gdf['Unmet_Demand'].fillna(0).values


def _pct(arr, q):
    return float(np.percentile(arr, q)) if len(arr) else float('nan')


for n in SCENARIOS:
    print(f"--- Simulando N={n} ---")

    # Línea base: estocástica → promediamos sobre múltiples iteraciones
    rng_map = np.random.default_rng(BASELINE_SEED + n)
    py_rng_map = random.Random(BASELINE_SEED + n)
    base_locs = generate_population_weighted_locations(n, gdf, rng=rng_map, py_rng=py_rng_map)

    baseline_metrics = []
    for it in range(BASELINE_ITERATIONS):
        seed_it = (BASELINE_SEED * 1_000_000) + (n * 10_000) + it
        locs_it = generate_population_weighted_locations(
            n, gdf,
            rng=np.random.default_rng(seed_it),
            py_rng=random.Random(seed_it)
        )
        baseline_metrics.append(evaluate_coverage(locs_it, gdf, radius_m=RADIUS_M))

    base_ev    = np.array([m['ev_population_served']  for m in baseline_metrics], dtype=float)
    base_unmet = np.array([m['unmet_demand_captured'] for m in baseline_metrics], dtype=float)
    base_eff   = np.array([m['coverage_efficiency']   for m in baseline_metrics], dtype=float)
    base_serv  = np.array([m['served_barrios_count']  for m in baseline_metrics], dtype=float)

    results.append({
        'N_Chargers': n,
        'Strategy': 'Baseline',
        'served_barrios_count':       float(base_serv.mean()),
        'ev_population_served':       float(base_ev.mean()),
        'unmet_demand_captured':      float(base_unmet.mean()),
        'coverage_efficiency':        float(base_eff.mean()),
        'baseline_iterations':        BASELINE_ITERATIONS,
        'ev_population_served_p05':   _pct(base_ev, 5),
        'ev_population_served_p50':   _pct(base_ev, 50),
        'ev_population_served_p95':   _pct(base_ev, 95),
        'unmet_demand_captured_p05':  _pct(base_unmet, 5),
        'unmet_demand_captured_p50':  _pct(base_unmet, 50),
        'unmet_demand_captured_p95':  _pct(base_unmet, 95),
        'coverage_efficiency_p05':    _pct(base_eff, 5),
        'coverage_efficiency_p50':    _pct(base_eff, 50),
        'coverage_efficiency_p95':    _pct(base_eff, 95),
    })

    for i, loc in enumerate(base_locs):
        locations_export.append({
            'Scenario_ID': f'Baseline_{n}', 'Type': 'Baseline',
            'N_Chargers': n, 'Lat': loc[0], 'Lng': loc[1], 'Hub_ID': i + 1
        })

    # Estrategia inteligente: determinista
    smart_locs = generate_smart_locations(n, X_coords, W_weights)
    smart_metrics = evaluate_coverage(smart_locs, gdf, radius_m=RADIUS_M)

    results.append({
        'N_Chargers': n,
        'Strategy': 'Smart',
        **smart_metrics,
        'baseline_iterations':         BASELINE_ITERATIONS,
        'ev_win_rate_vs_baseline':     float((smart_metrics['ev_population_served']  > base_ev).mean()),
        'unmet_win_rate_vs_baseline':  float((smart_metrics['unmet_demand_captured'] > base_unmet).mean()),
        'eff_win_rate_vs_baseline':    float((smart_metrics['coverage_efficiency']   > base_eff).mean()),
        'baseline_ev_mean':  float(base_ev.mean()),
        'baseline_ev_p05':   _pct(base_ev, 5),
        'baseline_ev_p50':   _pct(base_ev, 50),
        'baseline_ev_p95':   _pct(base_ev, 95),
    })

    for i, loc in enumerate(smart_locs):
        locations_export.append({
            'Scenario_ID': f'Smart_{n}', 'Type': 'Smart',
            'N_Chargers': n, 'Lat': loc[0], 'Lng': loc[1], 'Hub_ID': i + 1
        })

print("Simulación completada.")

--- Simulando N=10 ---
--- Simulando N=25 ---
--- Simulando N=50 ---
Simulación completada.


## Análisis de resultados y KPIs

In [4]:
df_res = pd.DataFrame(results)

df_pivot = df_res.pivot(
    index='N_Chargers',
    columns='Strategy',
    values=['ev_population_served', 'unmet_demand_captured', 'coverage_efficiency']
)

df_pivot['EV_Pop_Gain_Pct'] = (
    (df_pivot[('ev_population_served', 'Smart')] - df_pivot[('ev_population_served', 'Baseline')])
    / df_pivot[('ev_population_served', 'Baseline')] * 100
)
df_pivot['Demand_Gain_Pct'] = (
    (df_pivot[('unmet_demand_captured', 'Smart')] - df_pivot[('unmet_demand_captured', 'Baseline')])
    / df_pivot[('unmet_demand_captured', 'Baseline')] * 100
)

display(df_pivot)

PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

pd.DataFrame(locations_export).to_csv(
    os.path.join(PROCESSED_DIR, 'tableau_scenarios.csv'), index=False
)

df_kpis = df_res.copy()
df_kpis['Scenario_ID'] = df_kpis['Strategy'] + '_' + df_kpis['N_Chargers'].astype(str)
df_kpis.to_csv(os.path.join(PROCESSED_DIR, 'tableau_kpis.csv'), index=False)

gdf.drop(columns=['geometry', 'centroid'], errors='ignore').to_csv(
    os.path.join(PROCESSED_DIR, 'tableau_barrios_master.csv'), index=False
)

print("CSVs exportados para Tableau.")

ev_population_served               unmet_demand_captured  \
Strategy               Baseline         Smart              Baseline   
N_Chargers                                                            
10                  4465.551693   4991.346859             95.843969   
25                  9187.245127  13399.711479            199.763724   
50                 14245.151587  22575.682953            311.278521   

                       coverage_efficiency           EV_Pop_Gain_Pct  \
Strategy         Smart            Baseline     Smart                   
N_Chargers                                                             
10          118.286336            0.059217  0.066189       11.774473   
25          311.260463            0.121831  0.177691       45.851246   
50          536.528863            0.188903  0.299373       58.479766   

           Demand_Gain_Pct  
Strategy                    
N_Chargers                  
10               23.415524  
25               55.814307  
50               72.362957

CSVs exportados para Tableau.
